# Holy Wells in Ireland — a Wikidata exploration

This notebook queries [Wikidata](https://www.wikidata.org/) for Irish
holy wells and plots them on two maps: a marker map with images and
cross-references, and a hex-binned density grid showing where wells
cluster. The data model is a community effort of the
[WikiProject HolyWells](https://www.wikidata.org/wiki/Wikidata:WikiProject_HolyWells),
which links each well to its photograph on Wikimedia Commons and to
its corresponding feature in OpenStreetMap.

A browser-executable companion (`wikidata-holy-wells-map-live.qmd`)
runs the same pipeline directly in the browser via Pyodide.

## Requirements

```bash
pip install SPARQLWrapper pandas folium matplotlib
```


## About this notebook

### Why this dataset?

Irish holy wells are a near-ideal teaching dataset for linked open data:

- **Well-curated and citizen-scientist-driven** — the WikiProject has
  a consistent modelling schema (P31 = `holy well` + `holy well
  semantic concept`), so a short query returns a clean, meaningful
  slice of Wikidata.
- **Cross-referenced across three open data hubs** — each well is
  typically linked to Wikimedia Commons (image), OpenStreetMap (node
  or way), and sometimes to the Irish Sites and Monuments Record.
  This shows Linked Open Data doing its actual job: joining
  independently maintained datasets through shared identifiers.
- **Geographically clustered** — the Kilkenny concentration is a
  real-world feature (local mapping effort, not a coverage bias in
  the sense of a query artefact), which makes spatial aggregation
  didactically meaningful.

### Data-context notes

A few specifics about this particular query result worth flagging:

- The query returns **one row per image**, not one row per well. A
  well with three photographs produces three rows — so the first
  step after loading is to deduplicate on the Wikidata item URI.
- All three enrichment properties (`P18` image, `P625` coordinates,
  `P11693` OSM ID) are wrapped in `OPTIONAL`. Completeness varies:
  almost every well has coordinates, many have an OSM link, a
  substantial minority have a Commons image. The completeness
  analysis in Step 4 is part of the point of this notebook.
- Coordinates are returned as GeoSPARQL WKT literals in the form
  `Point(lon lat)` — note the longitude-first order, which differs
  from how mapping libraries expect inputs.

### Tooling notes

This local variant uses `SPARQLWrapper` for the query, `pandas` for
the DataFrame, `folium` for the maps, and `matplotlib` for the bar
chart. Setting a descriptive `User-Agent` is not optional for
Wikidata — the service rate-limits requests with a generic UA and
may block repeated anonymous access.


## Step 1 — Define the SPARQL query

The `P31` (*instance of*) filter uses two values joined by a comma,
which is SPARQL shorthand for *both* — a well must be typed as the
Wikidata class `Q1371047` (*holy well*) **and** the project's
controlled concept `Q126443332` (*holy well semantic concept*). The
two-class pattern is how the WikiProject distinguishes its curated
items from uncurated ones elsewhere in Wikidata.


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
USER_AGENT = "OER-HolyWells-Notebook/1.0"

SPARQL_QUERY = """
SELECT DISTINCT ?item ?itemLabel ?img ?geo ?osm WHERE {
  ?item wdt:P31 wd:Q1371047, wd:Q126443332.
  OPTIONAL { ?item wdt:P18 ?img. }
  OPTIONAL { ?item wdt:P625 ?geo. }
  OPTIONAL { ?item wdt:P11693 ?osm. }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

sparql = SPARQLWrapper(SPARQL_ENDPOINT, agent=USER_AGENT)
sparql.setQuery(SPARQL_QUERY)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
bindings = results["results"]["bindings"]
print(f"✓ Retrieved {len(bindings)} rows from Wikidata")

## Step 2 — Load the data

Two things happen here. First, we parse the WKT coordinate literal
with a case-insensitive regex — Wikidata writes `Point(lon lat)`
with a capital P, other endpoints use `POINT(...)`, and a defensive
parser handles both without per-endpoint code. Second, we group by
the Wikidata item URI so each well appears once, keeping the first
image and OSM ID found and the count of images as metadata.


In [ ]:
import re
import pandas as pd

assert bindings, "⚠ Please run the query cell first."

_WKT_POINT_RE = re.compile(
    r"point\s*\(\s*([-+\d.eE]+)\s+([-+\d.eE]+)\s*\)", re.IGNORECASE
)

def parse_wkt_point(wkt):
    """Parse 'Point(lon lat)' (any case) into (lat, lon)."""
    if not wkt:
        return (None, None)
    m = _WKT_POINT_RE.search(wkt)
    if not m:
        return (None, None)
    try:
        lon, lat = float(m.group(1)), float(m.group(2))
        return (lat, lon)
    except ValueError:
        return (None, None)

# Group rows by Wikidata item URI — a well with N images appears N times.
records = {}
for b in bindings:
    uri = b["item"]["value"]
    label = b.get("itemLabel", {}).get("value", "")
    img = b.get("img", {}).get("value")
    geo = b.get("geo", {}).get("value")
    osm = b.get("osm", {}).get("value")
    if uri not in records:
        lat, lon = parse_wkt_point(geo)
        records[uri] = {
            "item": uri,
            "qid": uri.rsplit("/", 1)[-1],
            "itemLabel": label,
            "latitude": lat,
            "longitude": lon,
            "osm": osm,
            "img_first": img,
            "img_count": 1 if img else 0,
        }
    else:
        if img:
            records[uri]["img_count"] += 1
            if not records[uri]["img_first"]:
                records[uri]["img_first"] = img

df = pd.DataFrame(records.values())
print(f"✓ {len(df)} unique wells ({len(bindings)} raw rows, "
      f"{len(bindings) - len(df)} were duplicates from multiple images)")
print(f"  with coordinates: {df['latitude'].notna().sum()}")
print(f"  with image:       {(df['img_count'] > 0).sum()}")
print(f"  with OSM ID:      {df['osm'].notna().sum()}")
df.head()

## Step 3 — Visualise

### Step 3a — Marker map with popups

Each well becomes a clickable circle marker. The popup shows the
well's name, a thumbnail of its Wikimedia Commons image (if one
exists), and hyperlinks to the Wikidata item and the OpenStreetMap
feature. This is the shape that Linked Open Data takes when it
reaches the reader: a single data point with navigable attachments
to multiple independently maintained open hubs.


In [ ]:
import folium
from folium import FeatureGroup

assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

geo = df[df["latitude"].notna()].copy()

def commons_thumb(img_url, width=220):
    """Turn a direct Commons File URL into a thumbnail URL via Special:FilePath."""
    if not img_url:
        return None
    filename = img_url.rsplit("/", 1)[-1]
    return (f"https://commons.wikimedia.org/wiki/Special:FilePath/"
            f"{filename}?width={width}")

centre = [geo["latitude"].mean(), geo["longitude"].mean()]
m = folium.Map(location=centre, zoom_start=8, tiles=None)

# Basemaps — same set as the browser variant
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer(
    "https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    name="OSM Humanitarian",
    attr="&copy; OpenStreetMap contributors, Tiles style by HOT",
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri World Imagery",
    attr="Tiles &copy; Esri",
    max_zoom=18,
).add_to(m)
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri World Terrain",
    attr="Tiles &copy; Esri",
    max_zoom=13,
).add_to(m)

with_img_layer    = FeatureGroup(name='<span style="display:inline-block;width:10px;height:10px;background:#1b7837;border-radius:50%;margin-right:6px;vertical-align:middle"></span>with image')
without_img_layer = FeatureGroup(name='<span style="display:inline-block;width:10px;height:10px;background:#8c8c8c;border-radius:50%;margin-right:6px;vertical-align:middle"></span>without image')

for _, row in geo.iterrows():
    popup_parts = [f'<b>{row["itemLabel"]}</b>']
    has_img = pd.notna(row["img_first"]) and bool(row["img_first"])
    if has_img:
        thumb = commons_thumb(row["img_first"], 220)
        popup_parts.append(
            f'<br><a href="{row["img_first"]}" target="_blank">'
            f'<img src="{thumb}" style="max-width:220px;max-height:160px;'
            f'margin-top:6px;border-radius:3px"></a>'
        )
        if row["img_count"] > 1:
            popup_parts.append(
                f'<br><small><i>{row["img_count"]} images on Commons</i></small>'
            )
    links = [f'<a href="{row["item"]}" target="_blank">Wikidata ({row["qid"]})</a>']
    if pd.notna(row["osm"]) and row["osm"]:
        osm_url = (f'https://www.openstreetmap.org/?mlat={row["latitude"]}'
                   f'&mlon={row["longitude"]}#map=19/{row["latitude"]}/{row["longitude"]}')
        links.append(f'<a href="{osm_url}" target="_blank">OSM ({row["osm"]})</a>')
    popup_parts.append(
        f'<br><small style="color:#555">{" &middot; ".join(links)}</small>'
    )
    colour = "#1b7837" if has_img else "#8c8c8c"
    marker = folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6, color=colour, weight=1.2,
        fill=True, fill_color=colour, fill_opacity=0.75,
        popup=folium.Popup("".join(popup_parts), max_width=260),
    )
    marker.add_to(with_img_layer if has_img else without_img_layer)

with_img_layer.add_to(m)
without_img_layer.add_to(m)
folium.LayerControl(collapsed=True).add_to(m)
m

### Step 3b — Hex-binned density grid

The marker map answers *where is each well*, but not *where do wells
cluster*. For that, we bin the points into a hexagonal grid and
colour each cell by count. Hex bins are built in pure Python —
no extra dependencies — and rendered as plain `folium.Polygon`
objects, which behave correctly under resizing, fullscreen and
print without any extra plumbing.

The projection step (`y = lat / cos(φ0)`) makes hexagons look
regular at the data's reference latitude without requiring a full
projection library.


In [ ]:
import math
from collections import Counter
from branca.element import MacroElement
from jinja2 import Template

assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

geo = df[df["latitude"].notna()].copy()
points = list(zip(geo["longitude"], geo["latitude"]))

# Adjust HEX_SIZE_DEG to taste. Ireland sits around 53°N, so 0.12°
# of longitude ≈ 8 km. Expect roughly 20–60 non-empty hexes for a
# dataset this size — if you get too many singletons, increase;
# if everything falls in a single cell, decrease.
HEX_SIZE_DEG = 0.12
phi0 = math.radians(geo["latitude"].mean())
ky   = 1.0 / math.cos(phi0)

def point_to_hex(lon, lat, size):
    x, y = lon, lat * ky
    q = (2/3) * x / size
    r = (-1/3) * x / size + (math.sqrt(3)/3) * y / size
    cx, cz = q, r; cy = -cx - cz
    rx, ry, rz = round(cx), round(cy), round(cz)
    dx, dy, dz = abs(rx - cx), abs(ry - cy), abs(rz - cz)
    if dx > dy and dx > dz:   rx = -ry - rz
    elif dy > dz:             ry = -rx - rz
    else:                     rz = -rx - ry
    return int(rx), int(rz)

def hex_polygon(q, r, size):
    cx = size * 1.5 * q
    cy = size * math.sqrt(3) * (r + q/2)
    return [((cy + size * math.sin(math.radians(60*i))) / ky,
              cx + size * math.cos(math.radians(60*i))) for i in range(6)]

counts = Counter(point_to_hex(lon, lat, HEX_SIZE_DEG) for lon, lat in points)
max_n = max(counts.values()) if counts else 1
ramp  = ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]

def bucket_colour(n):
    idx = min(len(ramp) - 1, int((n - 1) / max_n * len(ramp)))
    return ramp[idx]

# Build legend labels — integer count ranges per ramp step
edges = [int(math.ceil((i + 1) / len(ramp) * max_n)) for i in range(len(ramp))]
legend_items = []
prev = 1
for i, edge in enumerate(edges):
    if edge < prev:
        continue
    legend_items.append((ramp[i], f"{prev}" if edge == prev else f"{prev}–{edge}"))
    prev = edge + 1

print(f"✓ {len(counts)} non-empty hex cells "
      f"(max count per cell: {max_n}, total wells binned: {len(points)})")

centre = [geo["latitude"].mean(), geo["longitude"].mean()]
m2 = folium.Map(location=centre, zoom_start=8, tiles="OpenStreetMap")
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}",
    name="Esri World Terrain",
    attr="Tiles &copy; Esri",
    max_zoom=13,
).add_to(m2)

hex_layer = FeatureGroup(name="Hex density")
for (q, r), n in counts.items():
    folium.Polygon(
        locations=hex_polygon(q, r, HEX_SIZE_DEG),
        color="#555", weight=0.7,
        fill=True, fill_color=bucket_colour(n), fill_opacity=0.72,
        tooltip=f"<b>{n}</b> well{'s' if n != 1 else ''}",
    ).add_to(hex_layer)
hex_layer.add_to(m2)

# Legend — a small MacroElement with inline HTML
legend_html = ['<div style="position:absolute;bottom:20px;right:20px;z-index:9999;',
               'background:rgba(255,255,255,0.92);padding:6px 10px;border-radius:4px;',
               'font:12px/1.4 sans-serif;box-shadow:0 1px 3px rgba(0,0,0,0.15)">',
               '<b>Wells per hex</b><br>']
for colour, label in legend_items:
    legend_html.append(
        f'<span style="display:inline-block;width:14px;height:14px;'
        f'background:{colour};border:1px solid #999;margin-right:6px;'
        f'vertical-align:middle"></span>{label}<br>'
    )
legend_html.append("</div>")

class _Legend(MacroElement):
    _template = Template(
        "{% macro html(this, kwargs) %}"
        + "".join(legend_html)
        + "{% endmacro %}"
    )
m2.get_root().add_child(_Legend())

folium.LayerControl(collapsed=True).add_to(m2)
m2

## Step 4 — Explore

The DataFrame `df` stays in scope — feel free to modify the
cells below, add new ones, or start queries of your own. Two
starting points:

### Completeness of cross-references

How many wells are linked to each of the three open hubs? This is
the kind of question a *data steward* would ask before writing a
paper about the dataset: where are the gaps, what does "complete"
mean here, and is the completeness evenly distributed?


In [ ]:
import matplotlib.pyplot as plt

assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

completeness = {
    "has coordinates (P625)": df["latitude"].notna().sum(),
    "has Commons image (P18)": (df["img_count"] > 0).sum(),
    "has OSM ID (P11693)":     df["osm"].notna().sum(),
    "has all three":           (
        df["latitude"].notna() &
        (df["img_count"] > 0) &
        df["osm"].notna()
    ).sum(),
    "total wells":             len(df),
}

for k, v in completeness.items():
    pct = 100 * v / len(df)
    print(f"  {k:28s} {v:4d}   ({pct:5.1f}%)")

fig, ax = plt.subplots(figsize=(7, 3.2))
keys = list(completeness.keys())
vals = [completeness[k] for k in keys]
ax.barh(keys, vals, color=["#4C72B0","#55A868","#C44E52","#8172B2","#CCB974"])
ax.set_xlabel("number of wells")
ax.set_title("Completeness of Wikidata cross-references")
ax.invert_yaxis()
for i, v in enumerate(vals):
    ax.text(v + 2, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()

### Top image contributors

Which wells are the most thoroughly photographed? Multiple photos
on Commons usually mean the well is locally significant, easily
accessible, or has been the target of a focused documentation
effort. Counting images per well is a cheap proxy for *how much
attention a citizen-science community has paid to this object*.


In [ ]:
assert 'df' in dir() and not df.empty, "⚠ Please run the data-loading cell first."

top = (df[df["img_count"] > 0]
       .sort_values("img_count", ascending=False)
       .head(12)
       [["itemLabel", "img_count", "qid"]])

print(f"Top {len(top)} wells by number of images on Commons:\n")
for _, row in top.iterrows():
    print(f"  {row['img_count']:2d}  {row['itemLabel']:40s}  ({row['qid']})")

---

*Part of an Open Educational Resource series on knowledge graphs
and linked open data, produced in the context of
[NFDI4Objects](https://www.nfdi4objects.net/). Data: Wikidata
[WikiProject HolyWells](https://www.wikidata.org/wiki/Wikidata:WikiProject_HolyWells),
CC0. Tiles: OpenStreetMap contributors, Esri. Images: Wikimedia
Commons contributors.*
